In [37]:
import numpy as np
import os, json, re, glob
import torch
import numpy as np
from collections import defaultdict
import pickle
import torch.nn as nn
import torch.optim as optim
from collections import Counter



## Transition Function

In [38]:

# ── fixed grid definition ──────────────────────────────────────────────
GRID_WIDTH  = 9
GRID_HEIGHT = 9

# parse walls directly from grid_layout
# grid_layout[row][col]
grid_layout = [
    ["#","#","#","#","#","#","#","#","#"],  # row 0
    ["#","_","_","_","#","_","G","_","#"],  # row 1
    ["#","A","_","_","D","_","_","_","#"],  # row 2
    ["#","_","_","_","#","_","_","_","#"],  # row 3
    ["#","#","#","_","#","#","#","#","#"],  # row 4
    ["#","_","K","_","#","_","_","_","#"],  # row 5
    ["#","_","_","_","#","_","_","_","#"],  # row 6
    ["#","_","_","_","_","_","_","_","#"],  # row 7
    ["#","#","#","#","#","#","#","#","#"],  # row 8
]


In [39]:

# extract key positions
WALLS = set()
for row in range(GRID_HEIGHT):
    for col in range(GRID_WIDTH):
        if grid_layout[row][col] == "#":
            WALLS.add((col, row))

KEY_POS  = (2, 5)   # (col, row)
DOOR_POS = (4, 2)   # (col, row)
GOAL_POS = (6, 1)   # (col, row)

# all walkable cells = non-wall, non-goal cells
WALKABLE = set()
for row in range(GRID_HEIGHT):
    for col in range(GRID_WIDTH):
        if grid_layout[row][col] not in ("#",):
            WALKABLE.add((col, row))

print(f"Walls:    {len(WALLS)}")
print(f"Walkable: {len(WALKABLE)}")
print(f"Key:      {KEY_POS}")
print(f"Door:     {DOOR_POS}")
print(f"Goal:     {GOAL_POS}")


Walls:    42
Walkable: 39
Key:      (2, 5)
Door:     (4, 2)
Goal:     (6, 1)


In [40]:

# ── action deltas ──────────────────────────────────────────────────────
DELTAS = {
    'LEFT':  (-1,  0),
    'RIGHT': ( 1,  0),
    'UP':    ( 0, -1),   # row decreases going up
    'DOWN':  ( 0,  1),   # row increases going down
}

ACTIONS = ['LEFT', 'RIGHT', 'UP', 'DOWN']


In [41]:

# ── state space ────────────────────────────────────────────────────────
# state = (col, row, has_key, door_open)
# enumerate all valid states
def get_all_states():
    states = []
    for (col, row) in WALKABLE:
        for has_key in [False, True]:
            for door_open in [False, True]:
                # invalid: has_key=False but key already gone
                # (if agent has key, key is not on grid — both are consistent)
                # invalid: door_open=True but has_key=False
                # (can't open door without key)
                if door_open and not has_key:
                    continue  # door can't be open if agent never had key
                states.append((col, row, has_key, door_open))
    return states

ALL_STATES = get_all_states()
print(f"\nTotal valid states: {len(ALL_STATES)}")



Total valid states: 117


In [42]:

# ── transition function ────────────────────────────────────────────────
def f(state, action):
    """
    Deterministic transition function for this fixed grid.
    
    state  = (col, row, has_key, door_open)
    action = 'LEFT' | 'RIGHT' | 'UP' | 'DOWN'
    
    Returns next_state = (col, row, has_key, door_open)
    
    Rules:
      1. Compute candidate next (col, row) from action delta
      2. If candidate is a wall → stay (action has no effect)
      3. If candidate is the door AND door is locked → stay
      4. If candidate is the door AND door is open → pass through,
         door symbol is gone so cell is treated as empty
      5. If candidate is the key cell AND agent doesn't have key yet
         → pick up key (has_key becomes True)
      6. If agent steps onto door cell with key → door opens
         (door_open becomes True, agent moves to door cell)
      7. If agent reaches goal → terminal state, no further transitions
    """
    col, row, has_key, door_open = state

    # already at goal — terminal, no movement
    if (col, row) == GOAL_POS:
        return state

    dc, dr = DELTAS[action]
    next_col = col + dc
    next_row = row + dr

    # ── rule 1: out of bounds check ────────────────────────────────────
    if next_col < 0 or next_col >= GRID_WIDTH:
        return state
    if next_row < 0 or next_row >= GRID_HEIGHT:
        return state

    # ── rule 2: wall collision ─────────────────────────────────────────
    if (next_col, next_row) in WALLS:
        return state

    # ── rule 3: door collision when locked ────────────────────────────
    if (next_col, next_row) == DOOR_POS and not door_open:
        # agent cannot pass — stays in place
        return state

    # ── reaching this point: agent moves to (next_col, next_row) ──────
    new_has_key  = has_key
    new_door_open = door_open

    # ── rule 4: picking up key ─────────────────────────────────────────
    if (next_col, next_row) == KEY_POS and not has_key:
        new_has_key = True

    # ── rule 5: opening door ───────────────────────────────────────────
    # agent steps onto door cell with key → door opens
    # in minigrid the door opens when you step on it with key
    if (next_col, next_row) == DOOR_POS and has_key and not door_open:
        new_door_open = True

    return (next_col, next_row, new_has_key, new_door_open)


In [43]:


# ── verify transition function ─────────────────────────────────────────
print("\n=== Transition function verification ===\n")

# test 1: agent at start, goes RIGHT (should move to (2,2))
s = (1, 2, False, False)
print(f"Test 1 — start (1,2) + RIGHT:")
print(f"  {s} → {f(s, 'RIGHT')}")
print(f"  Expected: (2, 2, False, False)")

# test 2: agent tries to go through door (4,2) without key
s = (3, 2, False, False)
print(f"\nTest 2 — at (3,2) + RIGHT toward locked door at (4,2):")
print(f"  {s} → {f(s, 'RIGHT')}")
print(f"  Expected: (3, 2, False, False)  [blocked by locked door]")

# test 3: agent picks up key at (2,5)
s = (1, 5, False, False)
print(f"\nTest 3 — at (1,5) + RIGHT onto key at (2,5):")
print(f"  {s} → {f(s, 'RIGHT')}")
print(f"  Expected: (2, 5, True, False)  [key picked up]")

# test 4: agent has key, walks to door (4,2) → door opens
s = (3, 2, True, False)
print(f"\nTest 4 — at (3,2) with key + RIGHT onto door at (4,2):")
print(f"  {s} → {f(s, 'RIGHT')}")
print(f"  Expected: (4, 2, True, True)  [door opened]")

# test 5: agent has key, door is open, walks through door
s = (3, 2, True, True)
print(f"\nTest 5 — at (3,2) with key + door open + RIGHT through door:")
print(f"  {s} → {f(s, 'RIGHT')}")
print(f"  Expected: (4, 2, True, True)  [passes through]")

# test 6: wall collision
s = (1, 1, False, False)
print(f"\nTest 6 — at (1,1) + UP into wall at (1,0):")
print(f"  {s} → {f(s, 'UP')}")
print(f"  Expected: (1, 1, False, False)  [blocked by wall]")

# test 7: agent at (4,2) with key going RIGHT (room 2 side)
s = (4, 2, True, True)
print(f"\nTest 7 — at door cell (4,2) door open + RIGHT:")
print(f"  {s} → {f(s, 'RIGHT')}")
print(f"  Expected: (5, 2, True, True)")

# test 8: full path toward goal
print(f"\nTest 8 — path from (5,1) toward goal at (6,1):")
s = (5, 1, True, True)
print(f"  {s} + RIGHT → {f(s, 'RIGHT')}")
print(f"  Expected: (6, 1, True, True)  [reached goal]")



=== Transition function verification ===

Test 1 — start (1,2) + RIGHT:
  (1, 2, False, False) → (2, 2, False, False)
  Expected: (2, 2, False, False)

Test 2 — at (3,2) + RIGHT toward locked door at (4,2):
  (3, 2, False, False) → (3, 2, False, False)
  Expected: (3, 2, False, False)  [blocked by locked door]

Test 3 — at (1,5) + RIGHT onto key at (2,5):
  (1, 5, False, False) → (2, 5, True, False)
  Expected: (2, 5, True, False)  [key picked up]

Test 4 — at (3,2) with key + RIGHT onto door at (4,2):
  (3, 2, True, False) → (3, 2, True, False)
  Expected: (4, 2, True, True)  [door opened]

Test 5 — at (3,2) with key + door open + RIGHT through door:
  (3, 2, True, True) → (4, 2, True, True)
  Expected: (4, 2, True, True)  [passes through]

Test 6 — at (1,1) + UP into wall at (1,0):
  (1, 1, False, False) → (1, 1, False, False)
  Expected: (1, 1, False, False)  [blocked by wall]

Test 7 — at door cell (4,2) door open + RIGHT:
  (4, 2, True, True) → (5, 2, True, True)
  Expected: (5, 

In [44]:

# ── show full state space ──────────────────────────────────────────────
print(f"\n=== State space breakdown ===")
no_key_locked   = [(c,r,k,d) for c,r,k,d in ALL_STATES if not k and not d]
has_key_locked  = [(c,r,k,d) for c,r,k,d in ALL_STATES if k and not d]
has_key_open    = [(c,r,k,d) for c,r,k,d in ALL_STATES if k and d]

print(f"Phase 1 (no key, door locked):   {len(no_key_locked)} states")
print(f"Phase 2 (has key, door locked):  {len(has_key_locked)} states")
print(f"Phase 3 (has key, door open):    {len(has_key_open)} states")
print(f"Total:                           {len(ALL_STATES)} states")



=== State space breakdown ===
Phase 1 (no key, door locked):   39 states
Phase 2 (has key, door locked):  39 states
Phase 3 (has key, door open):    39 states
Total:                           117 states


In [45]:

# ── precompute full transition table ──────────────────────────────────
# T[state][action] = next_state
T = {}
for state in ALL_STATES:
    T[state] = {}
    for action in ACTIONS:
        T[state][action] = f(state, action)

print(f"\nTransition table computed: {len(T)} states × {len(ACTIONS)} actions")
print(f"= {len(T) * len(ACTIONS)} total transitions")


Transition table computed: 117 states × 4 actions
= 468 total transitions


## Phi Table -- Activations

In [46]:
# ── paths ──────────────────────────────────────────────────────────────
TRAJ_DIR  = r"C:\Users\user\Desktop\SPAR\trajectories"
ACT_DIR   = r"C:\Users\user\Desktop\SPAR\activations\activations_fixed_key_door_grid"
OUTPUT_DIR = r"C:\Users\user\Desktop\SPAR\phi_table"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LAYER  = "layer_15"   # use layer 15 — can repeat for 7 and 23
TOKEN_POS = "prompt_suffix"  # pre-reasoning activations

In [47]:
def parse_agent_pos(grid_state):
    for row_str in grid_state:
        m = re.match(r'^\s*(\d+)\s+', row_str)
        if not m:
            continue
        row_idx = int(m.group(1))
        cells = row_str[m.end():].split()
        for col_idx, cell in enumerate(cells):
            if cell == 'A':
                return (col_idx, row_idx)
    return None

In [48]:

def load_activation(act_stem, step_id, token_pos, layer):
    folder = os.path.join(ACT_DIR, act_stem, "openai__gpt-oss-20b",
                          layer, f"step_{step_id}", token_pos)
    if not os.path.exists(folder):
        return None
    files = sorted(os.listdir(folder), key=lambda f: int(f.split('.')[0]))
    if len(files) < 3:
        return None
    vecs = [torch.load(os.path.join(folder, file), map_location="cpu").float()
            for file in files[:3]]
    return torch.cat(vecs, dim=0).numpy()  # 8640-dim


In [49]:

# ── build phi_table ────────────────────────────────────────────────────
# For each unique state (col, row, has_key, door_open) seen in trajectories,
# collect ALL activations from all steps that were in that state,
# then average them to get one canonical phi(s).

# Structure:
# state_activations[(col, row, has_key, door_open)] = list of activation arrays
state_activations = defaultdict(list)
state_action_pairs = []   # (state, action) for IRL training

traj_files = sorted(glob.glob(os.path.join(TRAJ_DIR, "*.json")))
traj_files = traj_files[1:]
print(f"Found {len(traj_files)} trajectory files")

skipped = 0
collected = 0

for traj_path in traj_files:
    print(f"starting file {traj_path}")
    print(f"number of steps {len(traj['steps'])}")

    act_stem = os.path.splitext(os.path.basename(traj_path))[0]

    with open(traj_path, encoding='utf-8') as file:
        traj = json.load(file)

    step_count = 1
    for step in traj['steps']:
        print(f"step {step_count}")
        step_id  = step['step_id']
        action   = step['agent_action']
        has_key  = step['carrying_key']
        door_open = not any('D' in row for row in step['grid_state'])
        pos      = parse_agent_pos(step['grid_state'])
        step_count += 1
        if pos is None:
            skipped += 1
            continue

        state = (pos[0], pos[1], has_key, door_open)

        # load activation
        act = load_activation(act_stem, step_id, TOKEN_POS, LAYER)
        if act is None:
            skipped += 1
            continue

        state_activations[state].append(act)
        state_action_pairs.append((state, action))
        collected += 1

print(f"Collected {collected} samples, skipped {skipped}")
print(f"Unique states visited: {len(state_activations)}")


Found 100 trajectory files
starting file C:\Users\user\Desktop\SPAR\trajectories\together_ai_openai_gpt-oss-20b_rooms2_doorkey_grid0_traj0.json
number of steps 12
step 1
step 2
step 3
step 4
step 5
step 6
step 7
step 8
step 9
step 10
step 11
step 12
starting file C:\Users\user\Desktop\SPAR\trajectories\together_ai_openai_gpt-oss-20b_rooms2_doorkey_grid0_traj1.json
number of steps 12
step 1
step 2
step 3
step 4
step 5
step 6
step 7
step 8
step 9
step 10
step 11
step 12
step 13
step 14
step 15
step 16
step 17
step 18
step 19
starting file C:\Users\user\Desktop\SPAR\trajectories\together_ai_openai_gpt-oss-20b_rooms2_doorkey_grid0_traj10.json
number of steps 19
step 1
step 2
step 3
step 4
step 5
step 6
step 7
step 8
step 9
step 10
step 11
step 12
step 13
step 14
step 15
step 16
step 17
step 18
step 19
step 20
step 21
step 22
step 23
step 24
step 25
step 26
step 27
step 28
step 29
step 30
step 31
step 32
step 33
step 34
step 35
step 36
step 37
step 38
step 39
step 40
starting file C:\Users\

In [50]:

# ── show coverage ──────────────────────────────────────────────────────
print("\nState visit counts (top 20 most visited):")
sorted_states = sorted(state_activations.items(), key=lambda x: -len(x[1]))
for state, acts in sorted_states[:20]:
    col, row, hk, do = state
    print(f"  ({col},{row}) has_key={hk} door_open={do}: {len(acts)} visits")

print("\nStates with only 1 visit (uncertain phi):")
single_visit = [(s, a) for s, a in state_activations.items() if len(a) == 1]
print(f"  {len(single_visit)} states")



State visit counts (top 20 most visited):
  (5,5) has_key=False door_open=False: 552 visits
  (5,6) has_key=False door_open=False: 364 visits
  (6,5) has_key=False door_open=False: 131 visits
  (3,3) has_key=True door_open=False: 81 visits
  (2,5) has_key=True door_open=False: 77 visits
  (4,2) has_key=True door_open=True: 70 visits
  (3,5) has_key=True door_open=False: 69 visits
  (3,2) has_key=True door_open=True: 63 visits
  (3,4) has_key=True door_open=False: 62 visits
  (5,2) has_key=True door_open=True: 58 visits
  (4,7) has_key=False door_open=False: 56 visits
  (5,1) has_key=True door_open=True: 55 visits
  (6,6) has_key=False door_open=False: 52 visits
  (5,7) has_key=False door_open=False: 46 visits
  (3,5) has_key=False door_open=False: 40 visits
  (3,3) has_key=False door_open=False: 27 visits
  (3,4) has_key=False door_open=False: 27 visits
  (7,5) has_key=False door_open=False: 25 visits
  (3,6) has_key=False door_open=False: 23 visits
  (3,7) has_key=False door_open=Fal

In [51]:

# ── build phi_table by averaging activations per state ─────────────────
# Why average? Because the same state visited from different trajectories
# gives slightly different activations due to context. Averaging cancels
# out trajectory-specific noise and gives a cleaner state representation.
phi_table = {}
phi_std    = {}   # standard deviation — tells us how consistent the state rep is

for state, acts in state_activations.items():
    acts_array   = np.stack(acts, axis=0)   # shape (n_visits, 8640)
    phi_table[state] = acts_array.mean(axis=0)   # shape (8640,)
    phi_std[state]   = acts_array.std(axis=0)    # shape (8640,)

print(f"\nphi_table built: {len(phi_table)} states")



phi_table built: 54 states


In [52]:

# ── consistency check ──────────────────────────────────────────────────
# For states visited many times, how consistent is the activation?
# Low std = the activation reliably represents the state.
# High std = the activation varies a lot depending on trajectory context.
print("\nActivation consistency (mean std per state, grouped by visit count):")
for min_visits in [1, 2, 5, 10]:
    states_subset = [s for s, a in state_activations.items() if len(a) >= min_visits]
    if not states_subset:
        continue
    mean_stds = [phi_std[s].mean() for s in states_subset]
    print(f"  States with >={min_visits} visits: "
          f"n={len(states_subset)}, "
          f"mean activation std = {np.mean(mean_stds):.4f}")



Activation consistency (mean std per state, grouped by visit count):
  States with >=1 visits: n=54, mean activation std = 1.3765
  States with >=2 visits: n=49, mean activation std = 1.5170
  States with >=5 visits: n=40, mean activation std = 1.6332
  States with >=10 visits: n=29, mean activation std = 1.7247


In [53]:

# ── coverage check ─────────────────────────────────────────────────────
# Which states in the full state space did we NEVER see?
# These states can't be in phi_table and limit IRL coverage.
all_valid_states = []
for (col, row) in WALKABLE:
    for has_key in [False, True]:
        for door_open in [False, True]:
            if door_open and not has_key:
                continue  # physically impossible
            all_valid_states.append((col, row, has_key, door_open))

visited   = set(phi_table.keys())
unvisited = set(all_valid_states) - visited

print(f"\nFull state space: {len(all_valid_states)} states")
print(f"Visited (in phi_table): {len(visited)}")
print(f"Never visited: {len(unvisited)}")

if unvisited:
    print("\nUnvisited states (these need API calls to fill):")
    for s in sorted(unvisited)[:10]:
        print(f"  {s}")
    if len(unvisited) > 10:
        print(f"  ... and {len(unvisited)-10} more")



Full state space: 117 states
Visited (in phi_table): 54
Never visited: 63

Unvisited states (these need API calls to fill):
  (1, 1, True, False)
  (1, 1, True, True)
  (1, 2, True, False)
  (1, 2, True, True)
  (1, 3, True, False)
  (1, 3, True, True)
  (1, 5, True, False)
  (1, 5, True, True)
  (1, 6, True, False)
  (1, 6, True, True)
  ... and 53 more


In [54]:

# ── save everything ────────────────────────────────────────────────────
import pickle

save_data = {
    'phi_table':          phi_table,        # state → mean activation
    'phi_std':            phi_std,          # state → std of activations
    'state_action_pairs': state_action_pairs,  # [(state, action)] for IRL
    'state_visit_counts': {s: len(a) for s, a in state_activations.items()},
    'layer':              LAYER,
    'token_pos':          TOKEN_POS,
    'grid_info': {
        'walls':    list(WALLS),
        'walkable': list(WALKABLE),
        'key_pos':  KEY_POS,
        'door_pos': DOOR_POS,
        'goal_pos': GOAL_POS,
    }
}

save_path = os.path.join(OUTPUT_DIR, f"phi_table_{LAYER}_{TOKEN_POS}.pkl")
with open(save_path, 'wb') as file:
    pickle.dump(save_data, file)

print(f"\nSaved to: {save_path}")



Saved to: C:\Users\user\Desktop\SPAR\phi_table\phi_table_layer_15_prompt_suffix.pkl


In [55]:

# ── action distribution check ──────────────────────────────────────────
from collections import Counter
action_counts = Counter(a for _, a in state_action_pairs)
print(f"\nAction distribution in dataset:")
for action, count in sorted(action_counts.items()):
    print(f"  {action}: {count} ({100*count/len(state_action_pairs):.1f}%)")


Action distribution in dataset:
  DOWN: 376 (17.6%)
  LEFT: 636 (29.8%)
  RIGHT: 418 (19.6%)
  UP: 702 (32.9%)


### Phi_Table_Outputs

In [56]:

# ── load phi_table ─────────────────────────────────────────────────────
PHI_PATH  = r"C:\Users\user\Desktop\SPAR\phi_table\phi_table_layer_15_prompt_suffix.pkl"
SAVE_DIR  = r"C:\Users\user\Desktop\SPAR\phi_table"


In [57]:

with open(PHI_PATH, 'rb') as file:
    data = pickle.load(file)

phi_table_np        = data['phi_table']          # {state: np array (8640,)}
state_action_pairs  = data['state_action_pairs'] # [(state, action)]
visit_counts        = data['state_visit_counts']
grid_info           = data['grid_info']

WALLS    = set(map(tuple, grid_info['walls']))
KEY_POS  = tuple(grid_info['key_pos'])
DOOR_POS = tuple(grid_info['door_pos'])
GOAL_POS = tuple(grid_info['goal_pos'])
ACTIONS  = ['LEFT', 'RIGHT', 'UP', 'DOWN']
ACTION_TO_IDX = {a: i for i, a in enumerate(ACTIONS)}

print(f"phi_table states: {len(phi_table_np)}")
print(f"state-action pairs: {len(state_action_pairs)}")
print(f"activation dim: {next(iter(phi_table_np.values())).shape}")


phi_table states: 54
state-action pairs: 2132
activation dim: (8640,)


In [58]:

# ── convert phi_table to torch tensors ────────────────────────────────
# We need fast lookup: state → tensor
# Build a fixed index so we can do batch operations

state_list  = sorted(phi_table_np.keys())
state_to_idx = {s: i for i, s in enumerate(state_list)}
N_STATES     = len(state_list)
ACT_DIM      = 8640

# phi_matrix: shape (N_STATES, 8640)
phi_matrix = torch.tensor(
    np.stack([phi_table_np[s] for s in state_list], axis=0),
    dtype=torch.float32
)

# normalise — important for gradient stability
phi_mean = phi_matrix.mean(dim=0, keepdim=True)
phi_std  = phi_matrix.std(dim=0, keepdim=True) + 1e-8
phi_matrix_norm = (phi_matrix - phi_mean) / phi_std

print(f"\nphi_matrix shape: {phi_matrix_norm.shape}")



phi_matrix shape: torch.Size([54, 8640])


## Dataset

In [ ]:

# ── build training dataset ─────────────────────────────────────────────
# For each (s_t, a_t) pair where s_t is in phi_table:
#   - get phi of next state for all 4 actions
#   - the observed action is the label

# For each transition we store:
#   next_state_indices: shape (4,) — indices into state_list for each action's next state
#   action_idx:         int — index of observed action

next_state_indices = []   # (N_transitions, 4)
action_labels      = []   # (N_transitions,)
skipped = 0

for state, action in state_action_pairs:
    if state not in state_to_idx:
        skipped += 1
        continue

    # compute next state for all 4 actions
    row_indices = []
    valid = True
    for a in ACTIONS:
        ns = f(state, a)
        if ns not in state_to_idx:
            # next state not in phi_table — skip this transition
            valid = False
            break
        row_indices.append(state_to_idx[ns])

    if not valid:
        skipped += 1
        continue

    next_state_indices.append(row_indices)
    action_labels.append(ACTION_TO_IDX[action])

next_state_indices = torch.tensor(next_state_indices, dtype=torch.long)  # (N, 4)
action_labels      = torch.tensor(action_labels,      dtype=torch.long)  # (N,)
N_trans = len(action_labels)

print(f"\nTraining transitions: {N_trans}  (skipped {skipped})")
print(f"Action distribution:")
for a, idx in ACTION_TO_IDX.items():
    count = (action_labels == idx).sum().item()
    print(f"  {a}: {count} ({100*count/N_trans:.1f}%)")



Training transitions: 1822  (skipped 310)
Action distribution:
  LEFT: 627 (34.4%)
  RIGHT: 267 (14.7%)
  UP: 555 (30.5%)
  DOWN: 373 (20.5%)


## Linear Cost Model

In [60]:

# ── linear cost model ──────────────────────────────────────────────────
# C_theta(s) = theta^T phi(s)
# We learn theta as a single linear layer with no bias:
#   theta in R^8640
#
# Policy:
#   P(a|s) = softmax(-beta * C_theta(f(s,a)))_a
#           = softmax(-beta * theta^T phi(f(s,a)))_a
#
# Log-likelihood of observed action a_t:
#   log P(a_t|s_t) = log softmax(-beta * [C(f(s,a)) for a in A])[a_t]
#
# We MAXIMISE sum of log P(a_t|s_t) over all transitions
# = MINIMISE negative log-likelihood (cross-entropy loss)

class LinearCostIRL(nn.Module):
    def __init__(self, phi_dim, beta=1.0):
        super().__init__()
        # theta: the cost weights — initialised to zero
        self.theta = nn.Parameter(torch.zeros(phi_dim))
        self.beta  = beta

    def cost(self, phi):
        """
        phi: (batch, phi_dim) or (phi_dim,)
        returns scalar cost per state
        """
        return phi @ self.theta   # dot product

    def forward(self, phi_next_states):
        """
        phi_next_states: (N, 4, phi_dim) — phi of next state for each action
        returns: (N, 4) log probabilities over actions
        """
        # cost of each next state: (N, 4)
        costs = torch.einsum('naf,f->na', phi_next_states, self.theta)

        # policy: low cost = high probability
        # P(a|s) = softmax(-beta * costs)
        log_probs = torch.log_softmax(-self.beta * costs, dim=-1)
        return log_probs


In [61]:

# ── prepare batch tensor ───────────────────────────────────────────────
# For each transition, get phi of next states for all 4 actions
# phi_next: (N_trans, 4, 8640)
phi_next = phi_matrix_norm[next_state_indices]  # (N, 4, 8640)
print(f"\nphi_next shape: {phi_next.shape}")



phi_next shape: torch.Size([1822, 4, 8640])


In [62]:

# ── training ───────────────────────────────────────────────────────────
BETA        = 1.0
LR          = 0.001
N_EPOCHS    = 500
REG         = 0.01     # L2 regularisation on theta
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\nTraining on: {DEVICE}")

model     = LinearCostIRL(phi_dim=ACT_DIM, beta=BETA).to(DEVICE)
optimizer = optim.Adam([model.theta], lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

phi_next_dev   = phi_next.to(DEVICE)
labels_dev     = action_labels.to(DEVICE)

# baseline: random policy log-likelihood
baseline_ll = np.log(0.25)
print(f"Random baseline log-likelihood: {baseline_ll:.4f}")

best_ll    = -np.inf
best_theta = None
history    = []

for epoch in range(N_EPOCHS):
    model.train()
    optimizer.zero_grad()

    # forward pass
    log_probs = model(phi_next_dev)           # (N, 4)

    # NLL loss = -mean log P(observed action)
    nll = nn.functional.nll_loss(log_probs, labels_dev)

    # L2 regularisation on theta
    reg_loss = REG * (model.theta ** 2).sum()

    loss = nll + reg_loss
    loss.backward()
    optimizer.step()
    scheduler.step()

    # metrics
    with torch.no_grad():
        avg_ll   = -nll.item()
        preds    = log_probs.argmax(dim=-1)
        acc      = (preds == labels_dev).float().mean().item()

    history.append({'epoch': epoch, 'log_lik': avg_ll, 'acc': acc})

    if avg_ll > best_ll:
        best_ll    = avg_ll
        best_theta = model.theta.detach().cpu().clone()

    if epoch % 50 == 0 or epoch == N_EPOCHS - 1:
        print(f"  Epoch {epoch:4d}: "
              f"log-lik={avg_ll:.4f} (baseline={baseline_ll:.4f})  "
              f"acc={acc:.3f}  "
              f"|theta|={model.theta.norm().item():.4f}")



Training on: cpu
Random baseline log-likelihood: -1.3863
  Epoch    0: log-lik=-1.3863 (baseline=-1.3863)  acc=0.344  |theta|=0.0930
  Epoch   50: log-lik=-1.0272 (baseline=-1.3863)  acc=0.610  |theta|=0.5616
  Epoch  100: log-lik=-1.0142 (baseline=-1.3863)  acc=0.611  |theta|=0.6591
  Epoch  150: log-lik=-1.0110 (baseline=-1.3863)  acc=0.613  |theta|=0.7009
  Epoch  200: log-lik=-1.0097 (baseline=-1.3863)  acc=0.613  |theta|=0.7234
  Epoch  250: log-lik=-1.0091 (baseline=-1.3863)  acc=0.613  |theta|=0.7349
  Epoch  300: log-lik=-1.0088 (baseline=-1.3863)  acc=0.613  |theta|=0.7405
  Epoch  350: log-lik=-1.0086 (baseline=-1.3863)  acc=0.613  |theta|=0.7432
  Epoch  400: log-lik=-1.0086 (baseline=-1.3863)  acc=0.613  |theta|=0.7444
  Epoch  450: log-lik=-1.0085 (baseline=-1.3863)  acc=0.613  |theta|=0.7448
  Epoch  499: log-lik=-1.0085 (baseline=-1.3863)  acc=0.613  |theta|=0.7449


In [63]:

# ── restore best theta ─────────────────────────────────────────────────
model.theta.data.copy_(best_theta)
print(f"\nBest log-likelihood: {best_ll:.4f}")



Best log-likelihood: -1.0085


In [64]:

# ── evaluate ───────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    log_probs = model(phi_next_dev)
    preds     = log_probs.argmax(dim=-1).cpu()
    acc       = (preds == action_labels).float().mean().item()
    baseline  = action_labels.bincount().max().item() / N_trans

print(f"\n=== Final Results ===")
print(f"Action accuracy:  {acc:.3f}  (majority baseline: {baseline:.3f})")
print(f"Log-likelihood:   {best_ll:.4f}  (random baseline: {baseline_ll:.4f})")
print(f"Theta norm:       {best_theta.norm().item():.4f}")

print(f"\nPer-action accuracy:")
for a, idx in ACTION_TO_IDX.items():
    mask  = action_labels == idx
    if mask.sum() == 0:
        continue
    a_acc = (preds[mask] == action_labels[mask]).float().mean().item()
    print(f"  {a}: {a_acc:.3f}  (n={mask.sum().item()})")



=== Final Results ===
Action accuracy:  0.613  (majority baseline: 0.344)
Log-likelihood:   -1.0085  (random baseline: -1.3863)
Theta norm:       0.7449

Per-action accuracy:
  LEFT: 0.742  (n=627)
  RIGHT: 0.524  (n=267)
  UP: 0.780  (n=555)
  DOWN: 0.209  (n=373)


In [65]:

# ── cost analysis ──────────────────────────────────────────────────────
# compute cost of every state in phi_table
print(f"\n=== Cost function analysis ===")
with torch.no_grad():
    theta_cpu = best_theta
    all_costs = phi_matrix_norm @ theta_cpu  # (N_STATES,)

# group by phase
print("\nMean cost by task phase:")
for phase, mask_fn in [
    ("Phase 1 — no key, door locked",  lambda s: not s[2] and not s[3]),
    ("Phase 2 — has key, door locked", lambda s:     s[2] and not s[3]),
    ("Phase 3 — has key, door open",   lambda s:     s[2] and     s[3]),
]:
    indices = [i for i, s in enumerate(state_list) if mask_fn(s)]
    if not indices:
        continue
    phase_costs = all_costs[indices]
    print(f"  {phase}: mean={phase_costs.mean():.4f}  "
          f"min={phase_costs.min():.4f}  max={phase_costs.max():.4f}")

# cheapest and most expensive states
sorted_idx = all_costs.argsort()
print("\nTop 5 cheapest states (most desirable):")
for i in sorted_idx[:5]:
    s = state_list[i]
    print(f"  {s}: cost={all_costs[i].item():.4f}")

print("\nTop 5 most expensive states (least desirable):")
for i in sorted_idx[-5:]:
    s = state_list[i]
    print(f"  {s}: cost={all_costs[i].item():.4f}")



=== Cost function analysis ===

Mean cost by task phase:
  Phase 1 — no key, door locked: mean=4.3077  min=-2.0932  max=13.7867
  Phase 2 — has key, door locked: mean=-5.6911  min=-10.5379  max=-3.6573
  Phase 3 — has key, door open: mean=-10.9547  min=-16.1656  max=-3.6591

Top 5 cheapest states (most desirable):
  (5, 2, True, True): cost=-16.1656
  (5, 1, True, True): cost=-15.6162
  (4, 2, True, True): cost=-13.8027
  (3, 2, True, True): cost=-12.6725
  (6, 2, True, True): cost=-11.8740

Top 5 most expensive states (least desirable):
  (7, 5, False, False): cost=7.1455
  (6, 7, False, False): cost=8.1255
  (7, 6, False, False): cost=8.4466
  (7, 7, False, False): cost=11.5175
  (1, 7, False, False): cost=13.7867


In [66]:

# ── save model ─────────────────────────────────────────────────────────
import pickle
save_path = os.path.join(SAVE_DIR, "irl_linear_cost_layer15.pkl")
with open(save_path, 'wb') as f:
    pickle.dump({
        'theta':     best_theta.numpy(),
        'phi_mean':  phi_mean.numpy(),
        'phi_std':   phi_std.numpy(),
        'beta':      BETA,
        'history':   history,
        'state_list': state_list,
        'best_ll':   best_ll,
        'final_acc': acc,
    }, f)
print(f"\nModel saved to: {save_path}")


Model saved to: C:\Users\user\Desktop\SPAR\phi_table\irl_linear_cost_layer15.pkl


## NN

In [67]:
# ── two-layer neural network cost model ───────────────────────────────
# C_theta(s) = g_theta(phi(s))
# where g_theta is: Linear(8640→128) → ReLU → Linear(128→1)
#
# Policy stays the same:
#   P(a|s) = softmax(-beta * C_theta(f(s,a)))_a
#
# The difference from linear:
#   Linear: cost = theta^T phi(s)          — one weight per activation dim
#   MLP:    cost = W2 ReLU(W1 phi(s) + b1) — can capture nonlinear patterns

class MLPCostIRL(nn.Module):
    def __init__(self, phi_dim, hidden_dim=128, beta=1.0, dropout=0.1):
        super().__init__()
        self.beta = beta
        self.net = nn.Sequential(
            nn.Linear(phi_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),   # outputs scalar cost
        )
        # initialise final layer to near-zero so costs start small
        nn.init.xavier_uniform_(self.net[0].weight, gain=0.1)
        nn.init.xavier_uniform_(self.net[3].weight, gain=0.1)
        nn.init.zeros_(self.net[6].weight)

    def cost(self, phi):
        """
        phi: (..., phi_dim)
        returns: (...) scalar cost per state
        """
        return self.net(phi).squeeze(-1)

    def forward(self, phi_next_states):
        """
        phi_next_states: (N, 4, phi_dim)
        returns: (N, 4) log probabilities
        """
        costs     = self.cost(phi_next_states)          # (N, 4)
        log_probs = torch.log_softmax(-self.beta * costs, dim=-1)
        return log_probs


In [68]:

# ── training ───────────────────────────────────────────────────────────
BETA       = 1.0
LR         = 0.001
N_EPOCHS   = 500
REG        = 0.01
HIDDEN_DIM = 128
DROPOUT    = 0.1
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Training MLP cost model on: {DEVICE}")
print(f"Architecture: {ACT_DIM} → {HIDDEN_DIM} → {HIDDEN_DIM} → 1")
print(f"Random baseline log-likelihood: {np.log(0.25):.4f}\n")

model     = MLPCostIRL(phi_dim=ACT_DIM, hidden_dim=HIDDEN_DIM,
                       beta=BETA, dropout=DROPOUT).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=REG)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

phi_next_dev = phi_next.to(DEVICE)
labels_dev   = action_labels.to(DEVICE)

# count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

best_ll    = -np.inf
best_state = None
history    = []

for epoch in range(N_EPOCHS):
    model.train()
    optimizer.zero_grad()

    log_probs = model(phi_next_dev)
    nll       = nn.functional.nll_loss(log_probs, labels_dev)
    loss      = nll   # weight_decay in Adam handles regularisation

    loss.backward()

    # gradient clipping — important for MLP stability
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    scheduler.step()

    with torch.no_grad():
        avg_ll = -nll.item()
        preds  = log_probs.argmax(dim=-1)
        acc    = (preds == labels_dev).float().mean().item()

    history.append({'epoch': epoch, 'log_lik': avg_ll, 'acc': acc})

    if avg_ll > best_ll:
        best_ll    = avg_ll
        best_state = {k: v.cpu().clone()
                      for k, v in model.state_dict().items()}

    if epoch % 50 == 0 or epoch == N_EPOCHS - 1:
        print(f"Epoch {epoch:4d}: "
              f"log-lik={avg_ll:.4f}  "
              f"acc={acc:.3f}")


Training MLP cost model on: cpu
Architecture: 8640 → 128 → 128 → 1
Random baseline log-likelihood: -1.3863

Trainable parameters: 1,122,689
Epoch    0: log-lik=-1.3863  acc=0.344
Epoch   50: log-lik=-1.0306  acc=0.523
Epoch  100: log-lik=-1.0283  acc=0.540
Epoch  150: log-lik=-1.0248  acc=0.541
Epoch  200: log-lik=-1.0225  acc=0.543
Epoch  250: log-lik=-1.0275  acc=0.532
Epoch  300: log-lik=-1.0196  acc=0.541
Epoch  350: log-lik=-1.0190  acc=0.532
Epoch  400: log-lik=-1.0167  acc=0.546
Epoch  450: log-lik=-1.0149  acc=0.540
Epoch  499: log-lik=-1.0207  acc=0.521


In [69]:

# restore best
model.load_state_dict(best_state)
print(f"\nBest log-likelihood: {best_ll:.4f}")



Best log-likelihood: -1.0113


In [70]:

# ── final evaluation ───────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    log_probs = model(phi_next_dev)
    preds     = log_probs.argmax(dim=-1).cpu()
    acc       = (preds == action_labels).float().mean().item()
    baseline  = action_labels.bincount().max().item() / N_trans

print(f"\n=== Final Results (MLP) ===")
print(f"Action accuracy:  {acc:.3f}  (majority baseline: {baseline:.3f})")
print(f"Best log-lik:     {best_ll:.4f}  (random baseline: {np.log(0.25):.4f})")

print(f"\nPer-action accuracy:")
for a, idx in ACTION_TO_IDX.items():
    mask = action_labels == idx
    if mask.sum() == 0:
        continue
    a_acc = (preds[mask] == action_labels[mask]).float().mean().item()
    print(f"  {a:6s}: {a_acc:.3f}  (n={mask.sum().item()})")



=== Final Results (MLP) ===
Action accuracy:  0.613  (majority baseline: 0.344)
Best log-lik:     -1.0113  (random baseline: -1.3863)

Per-action accuracy:
  LEFT  : 0.742  (n=627)
  RIGHT : 0.524  (n=267)
  UP    : 0.780  (n=555)
  DOWN  : 0.209  (n=373)


In [71]:

# ── cost analysis ──────────────────────────────────────────────────────
print(f"\n=== Cost function analysis (MLP) ===")
model.eval()
with torch.no_grad():
    phi_all   = phi_matrix_norm.to(DEVICE)
    all_costs = model.cost(phi_all).cpu()   # (N_STATES,)

print("\nMean cost by task phase:")
for phase, mask_fn in [
    ("Phase 1 — no key, door locked",  lambda s: not s[2] and not s[3]),
    ("Phase 2 — has key, door locked", lambda s:     s[2] and not s[3]),
    ("Phase 3 — has key, door open",   lambda s:     s[2] and     s[3]),
]:
    indices = [i for i, s in enumerate(state_list) if mask_fn(s)]
    if not indices:
        continue
    phase_costs = all_costs[torch.tensor(indices)]
    print(f"  {phase}:")
    print(f"    mean={phase_costs.mean():.4f}  "
          f"min={phase_costs.min():.4f}  "
          f"max={phase_costs.max():.4f}")

sorted_idx = all_costs.argsort()
print("\nTop 5 cheapest states (most desirable):")
for i in sorted_idx[:5].tolist():
    s = state_list[i]
    print(f"  {s}: cost={all_costs[i].item():.4f}")

print("\nTop 5 most expensive states (least desirable):")
for i in sorted_idx[-5:].tolist():
    s = state_list[i]
    print(f"  {s}: cost={all_costs[i].item():.4f}")



=== Cost function analysis (MLP) ===

Mean cost by task phase:
  Phase 1 — no key, door locked:
    mean=0.5704  min=-4.3565  max=5.9101
  Phase 2 — has key, door locked:
    mean=-7.5244  min=-13.6943  max=2.4674
  Phase 3 — has key, door open:
    mean=-1.4885  min=-8.7835  max=2.8427

Top 5 cheapest states (most desirable):
  (3, 3, True, False): cost=-13.6943
  (2, 3, True, False): cost=-11.3123
  (3, 4, True, False): cost=-9.5863
  (2, 3, True, True): cost=-8.7835
  (2, 6, True, False): cost=-8.4866

Top 5 most expensive states (least desirable):
  (6, 7, False, False): cost=3.2985
  (7, 6, False, False): cost=3.4730
  (7, 3, False, False): cost=4.0545
  (1, 7, False, False): cost=4.3818
  (7, 7, False, False): cost=5.9101


In [72]:

# ── compare linear vs MLP ──────────────────────────────────────────────
# load linear results if saved
linear_path = os.path.join(SAVE_DIR, "irl_linear_cost_layer15.pkl")
if os.path.exists(linear_path):
    with open(linear_path, 'rb') as fh:
        linear_data = pickle.load(fh)
    print(f"\n=== Comparison ===")
    print(f"{'Model':10s}  {'Accuracy':>10s}  {'Log-lik':>10s}")
    print(f"{'Linear':10s}  {linear_data['final_acc']:>10.3f}  "
          f"{linear_data['best_ll']:>10.4f}")
    print(f"{'MLP':10s}  {acc:>10.3f}  {best_ll:>10.4f}")
    print(f"{'Random':10s}  {'0.250':>10s}  {np.log(0.25):>10.4f}")



=== Comparison ===
Model         Accuracy     Log-lik
Linear           0.613     -1.0085
MLP              0.613     -1.0113
Random           0.250     -1.3863


In [74]:

# ── save ───────────────────────────────────────────────────────────────
with open(os.path.join(SAVE_DIR, "irl_mlp_cost_layer15.pkl"), 'wb') as fh:
    pickle.dump({
        'model_state_dict': best_state,
        'phi_mean':         phi_mean.numpy(),
        'phi_std':          phi_std.numpy(),
        'hidden_dim':       HIDDEN_DIM,
        'beta':             BETA,
        'history':          history,
        'state_list':       state_list,
        'best_ll':          best_ll,
        'final_acc':        acc,
    }, fh)

print(f"\nSaved → {SAVE_DIR}/irl_mlp_cost_layer15.pkl")


Saved → C:\Users\user\Desktop\SPAR\phi_table/irl_mlp_cost_layer15.pkl
